# FT-Transformer on Jannis & Higgs — TensorFlow port vs. the paper (Gorishniy et al., NeurIPS 2021)

Self-contained: **no repo cloning**. The data are downloaded straight from OpenML, the model/training code is in this notebook, and the paper's tuned hyper-parameters are embedded.

| Dataset | OpenML link | Paper test accuracy (FT-Transformer, tuned, 15 seeds) |
|---|---|---|
| Jannis (83,733 × 54, 4 classes) | https://www.openml.org/d/41168 | **0.732** |
| Higgs Small (98,050 × 28, binary) | https://www.openml.org/d/23512 | **0.729** |

**Splits.** The authors' data archive is hosted on Dropbox. This notebook instead builds the split itself: `train_test_split(test_size=0.2)` twice, which reproduces the paper's Table 7 sizes exactly (Jannis 53,588 / 13,398 / 16,747; Higgs 62,752 / 15,688 / 19,610). The *rows* in each part are not guaranteed to be the authors' rows, so absolute numbers can differ from the paper by roughly the test-set sampling error (≈ ±0.003). The comparison between the TF port and the official PyTorch code, run on **the same split**, is not affected by this.


Use a **GPU runtime** (Runtime → Change runtime type → T4). Start with `SEEDS = [0]` to measure speed.

In [1]:
# ===================== settings =====================
DATASETS = ["jannis", "higgs_small"]
SEEDS = list(range(5))          # the paper used 15 seeds (0..14); start small and scale up
RUN_TENSORFLOW_PORT = True
RUN_OFFICIAL_PYTORCH = True     # optional: needs rtdl-revisiting-models-main.zip uploaded to /content (skipped if absent)
SAVE_TO_DRIVE = False           # True -> results survive a Colab disconnect (finished seeds are skipped on re-run)
SMOKE_TEST = False              # True -> 2 epochs + small batch: only checks that the pipeline runs

# Paper (Table 2, tuned FT-Transformer, 15 seeds): test accuracy mean, std, and authors' V100 minutes per run
PAPER = {"jannis": (0.7323, 0.0020, 9.6), "higgs_small": (0.7290, 0.0015, 4.3)}

# Tuned hyper-parameters from the paper's repo (output/<dataset>/ft_transformer/tuned/0.toml)
CFG = {
    "jannis": {
        "model": dict(n_layers=4, d_token=288, n_heads=8, d_ffn_factor=1.061031416526688, attention_dropout=0.4194914895568438,
                      ffn_dropout=0.4632881116664471, residual_dropout=0.0, prenormalization=False,
                      activation="reglu", initialization="kaiming"),
        "training": dict(batch_size=512, eval_batch_size=8192, lr=0.0001089430445312344, weight_decay=2.897993930091118e-05, patience=16),
    },
    "higgs_small": {
        "model": dict(n_layers=2, d_token=264, n_heads=8, d_ffn_factor=2.44721762662163, attention_dropout=0.2285776112227358,
                      ffn_dropout=0.3224216086083693, residual_dropout=0.0, prenormalization=True,
                      activation="reglu", initialization="kaiming"),
        "training": dict(batch_size=512, eval_batch_size=8192, lr=2.520396383745548e-05, weight_decay=1.871343748594096e-05, patience=16),
    },
}

In [2]:
import os, sys, re, json, time, shutil, subprocess
from pathlib import Path
import numpy as np, pandas as pd
os.environ.setdefault("TF_FORCE_GPU_ALLOW_GROWTH", "true")   # lets the optional PyTorch subprocess share the GPU

ROOT = Path(os.environ.get("FT_ROOT", "/content"))
DATA = ROOT / "data"
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = Path("/content/drive/MyDrive/ft_repro")
else:
    OUT = ROOT / "ft_repro"
OUT.mkdir(parents=True, exist_ok=True)

def sh(cmd, **kw):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True, **kw)

subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True)

CompletedProcess(args='nvidia-smi --query-gpu=name,memory.total --format=csv,noheader', returncode=0)

## 1. Data — downloaded from OpenML, split to the paper's Table 7 sizes

In [3]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

SOURCES = {
    "jannis":      dict(openml_id=41168, n=83733, k=54, n_classes=4, task="multiclass", url="https://www.openml.org/d/41168"),
    "higgs_small": dict(openml_id=23512, n=98050, k=28, n_classes=2, task="binclass",   url="https://www.openml.org/d/23512"),
}
PAPER_SPLIT = {"jannis": (53588, 13398, 16747), "higgs_small": (62752, 15688, 19610)}

def build_dataset(ds):
    s, out = SOURCES[ds], DATA / ds
    if (out / "info.json").exists():
        return
    try:
        d = fetch_openml(data_id=s["openml_id"], as_frame=True, parser="auto")
    except Exception as e:
        raise RuntimeError(f"OpenML download failed for {ds} ({s['url']}): {e}\n"
                           f"Re-run the cell; if it keeps failing, OpenML may be temporarily down.") from e
    X = d.data.to_numpy(dtype="float32")
    y = pd.Categorical(d.target).codes.astype("int64")
    assert X.shape == (s["n"], s["k"]) and len(np.unique(y)) == s["n_classes"], \
        f"unexpected {ds} shape {X.shape} / {len(np.unique(y))} classes - wrong OpenML version?"
    idx = np.arange(len(y))
    tv, te = train_test_split(idx, test_size=0.2, random_state=0, stratify=y)
    tr, va = train_test_split(tv, test_size=0.2, random_state=0, stratify=y[tv])
    assert (len(tr), len(va), len(te)) == PAPER_SPLIT[ds]
    out.mkdir(parents=True, exist_ok=True)
    for part, ix in [("train", tr), ("val", va), ("test", te)]:       # same file layout the official code reads
        np.save(out / f"N_{part}.npy", X[ix]); np.save(out / f"y_{part}.npy", y[ix])
    (out / "info.json").write_text(json.dumps({
        "name": ds, "id": f"openml-{s['openml_id']}", "task_type": s["task"], "n_num_features": s["k"], "n_cat_features": 0,
        "train_size": len(tr), "val_size": len(va), "test_size": len(te), "n_classes": s["n_classes"]}))

for ds in DATASETS:
    build_dataset(ds)
    print(ds, json.loads((DATA / ds / "info.json").read_text()))

jannis {'name': 'jannis', 'id': 'openml-41168', 'task_type': 'multiclass', 'n_num_features': 54, 'n_cat_features': 0, 'train_size': 53588, 'val_size': 13398, 'test_size': 16747, 'n_classes': 4}
higgs_small {'name': 'higgs_small', 'id': 'openml-23512', 'task_type': 'binclass', 'n_num_features': 28, 'n_cat_features': 0, 'train_size': 62752, 'val_size': 15688, 'test_size': 19610, 'n_classes': 2}


## 2. Model + training code (TensorFlow / Keras port of the paper's FT-Transformer)

In [4]:
from pathlib import Path
Path('ft_tf.py').write_text(r'''"""FT-Transformer in TensorFlow/Keras 3, mirroring bin/ft_transformer.py of
yandex-research/tabular-dl-revisiting-models (numerical features only)."""
import math
import numpy as np
import tensorflow as tf
import keras
from keras import layers, initializers


def torch_kernel_init():
    # torch.nn.Linear weight init: U(-1/sqrt(fan_in), 1/sqrt(fan_in))
    return initializers.VarianceScaling(scale=1.0 / 3.0, mode="fan_in", distribution="uniform")


def torch_bias_init(fan_in):
    b = 1.0 / math.sqrt(fan_in)
    return initializers.RandomUniform(-b, b)


def dense(units, fan_in, zero_bias=False):
    return layers.Dense(
        units,
        kernel_initializer=torch_kernel_init(),
        bias_initializer="zeros" if zero_bias else torch_bias_init(fan_in),
    )


class Tokenizer(layers.Layer):
    """[CLS] token + one embedding per numerical feature: x_j * W_j + b_j."""

    def __init__(self, d_numerical, d_token, bias=True, **kw):
        super().__init__(**kw)
        self.d_numerical, self.d_token, self.use_bias = d_numerical, d_token, bias

    def build(self, input_shape):
        b = 1.0 / math.sqrt(self.d_token)  # kaiming_uniform_(a=sqrt(5)) on (rows, d_token)
        init = initializers.RandomUniform(-b, b)
        self.weight = self.add_weight(shape=(self.d_numerical + 1, self.d_token), initializer=init, name="weight")
        self.bias = (
            self.add_weight(shape=(self.d_numerical, self.d_token), initializer=init, name="bias")
            if self.use_bias else None
        )

    def call(self, x_num):
        ones = tf.ones_like(x_num[:, :1])
        x = tf.concat([ones, x_num], axis=1)                    # [B, k+1]
        x = self.weight[None] * x[:, :, None]                   # [B, k+1, d]
        if self.bias is not None:
            bias = tf.concat([tf.zeros_like(self.bias[:1]), self.bias], axis=0)  # no bias for [CLS]
            x = x + bias[None]
        return x


class MultiheadAttention(layers.Layer):
    def __init__(self, d, n_heads, dropout, **kw):
        super().__init__(**kw)
        assert d % n_heads == 0 and n_heads > 1
        self.d, self.n_heads = d, n_heads
        self.W_q, self.W_k, self.W_v = (dense(d, d, zero_bias=True) for _ in range(3))
        self.W_out = dense(d, d, zero_bias=True)
        self.drop = layers.Dropout(dropout) if dropout else None

    def _split(self, x):
        b, t = tf.shape(x)[0], tf.shape(x)[1]
        x = tf.reshape(x, [b, t, self.n_heads, self.d // self.n_heads])
        return tf.transpose(x, [0, 2, 1, 3])                    # [B, H, T, dh]

    def call(self, x_q, x_kv, training=False):
        q, k, v = self._split(self.W_q(x_q)), self._split(self.W_k(x_kv)), self._split(self.W_v(x_kv))
        dh = self.d // self.n_heads
        attn = tf.nn.softmax(tf.matmul(q, k, transpose_b=True) / math.sqrt(dh), axis=-1)
        if self.drop is not None:
            attn = self.drop(attn, training=training)
        x = tf.matmul(attn, v)                                  # [B, H, Tq, dh]
        x = tf.transpose(x, [0, 2, 1, 3])
        x = tf.reshape(x, [tf.shape(x)[0], tf.shape(x)[1], self.d])
        return self.W_out(x)


def reglu(x):
    a, b = tf.split(x, 2, axis=-1)
    return a * tf.nn.relu(b)


class FTTransformer(keras.Model):
    def __init__(self, d_numerical, d_out, n_layers, d_token, n_heads, d_ffn_factor,
                 attention_dropout, ffn_dropout, residual_dropout, prenormalization, **kw):
        super().__init__(**kw)
        self.prenorm, self.ffn_drop, self.res_drop = prenormalization, ffn_dropout, residual_dropout
        self.tokenizer = Tokenizer(d_numerical, d_token)
        d_hidden = int(d_token * d_ffn_factor)
        self.blocks = []
        for i in range(n_layers):
            blk = dict(
                attention=MultiheadAttention(d_token, n_heads, attention_dropout),
                linear0=dense(d_hidden * 2, d_token),           # ReGLU doubles the width
                linear1=dense(d_token, d_hidden),
                norm1=layers.LayerNormalization(epsilon=1e-5),
            )
            if not prenormalization or i:                       # first prenorm layer has no norm0
                blk["norm0"] = layers.LayerNormalization(epsilon=1e-5)
            self.blocks.append(blk)
        self.last_norm = layers.LayerNormalization(epsilon=1e-5) if prenormalization else None
        self.head = dense(d_out, d_token)
        self.d_out = d_out
        self.f_drop = layers.Dropout(ffn_dropout) if ffn_dropout else None
        self.r_drop = layers.Dropout(residual_dropout) if residual_dropout else None

    def _start(self, x, blk, key):
        return blk[key](x) if (self.prenorm and key in blk) else x

    def _end(self, x, res, blk, key, training):
        if self.r_drop is not None:
            res = self.r_drop(res, training=training)
        x = x + res
        return x if self.prenorm else blk[key](x)

    def call(self, x_num, training=False):
        x = self.tokenizer(x_num)
        for i, blk in enumerate(self.blocks):
            last = i + 1 == len(self.blocks)
            r = self._start(x, blk, "norm0")
            r = blk["attention"](r[:, :1] if last else r, r, training=training)  # last layer: [CLS] only
            if last:
                x = x[:, : tf.shape(r)[1]]
            x = self._end(x, r, blk, "norm0", training)
            r = self._start(x, blk, "norm1")
            r = reglu(blk["linear0"](r))
            if self.f_drop is not None:
                r = self.f_drop(r, training=training)
            r = blk["linear1"](r)
            x = self._end(x, r, blk, "norm1", training)
        x = x[:, 0]
        if self.last_norm is not None:
            x = self.last_norm(x)
        x = tf.nn.relu(x)                                       # last_activation for ReGLU
        x = self.head(x)
        return tf.squeeze(x, -1) if self.d_out == 1 else x


# ----------------------------------------------------------------------------
# Parameter naming (same names as the official PyTorch model) + weight-decay rule
# ----------------------------------------------------------------------------
def named_parameters(m):
    """[(torch_name, tf_variable, is_dense_kernel)] in the official PyTorch order."""
    P = [("tokenizer.weight", m.tokenizer.weight, False)]
    if m.tokenizer.bias is not None:
        P.append(("tokenizer.bias", m.tokenizer.bias, False))
    for i, b in enumerate(m.blocks):
        for k in ["W_q", "W_k", "W_v", "W_out"]:
            l = getattr(b["attention"], k)
            P += [(f"layers.{i}.attention.{k}.weight", l.kernel, True), (f"layers.{i}.attention.{k}.bias", l.bias, False)]
        for k in ["linear0", "linear1"]:
            P += [(f"layers.{i}.{k}.weight", b[k].kernel, True), (f"layers.{i}.{k}.bias", b[k].bias, False)]
        for k in ["norm0", "norm1"]:
            if k in b:
                P += [(f"layers.{i}.{k}.weight", b[k].gamma, False), (f"layers.{i}.{k}.bias", b[k].beta, False)]
    if m.last_norm is not None:
        P += [("last_normalization.weight", m.last_norm.gamma, False), ("last_normalization.bias", m.last_norm.beta, False)]
    P += [("head.weight", m.head.kernel, True), ("head.bias", m.head.bias, False)]
    return P


def needs_wd(name):
    # identical to the official code (note: 'last_normalization.weight' DOES get weight decay there)
    return all(x not in name for x in ["tokenizer", ".norm", ".bias"])


class TorchAdamW:
    """AdamW with exactly torch.optim.AdamW's update rule (decoupled decay, eps outside the bias correction)."""

    def __init__(self, variables, decay_flags, lr, weight_decay, betas=(0.9, 0.999), eps=1e-8):
        self.vars, self.lr, self.b1, self.b2, self.eps = variables, lr, betas[0], betas[1], eps
        self.wd = [weight_decay if f else 0.0 for f in decay_flags]
        self.m = [tf.Variable(tf.zeros_like(v), trainable=False) for v in variables]
        self.v = [tf.Variable(tf.zeros_like(v), trainable=False) for v in variables]
        self.t = tf.Variable(0.0, trainable=False)

    def apply(self, grads):
        self.t.assign_add(1.0)
        bc1 = 1.0 - tf.pow(self.b1, self.t)
        bc2 = 1.0 - tf.pow(self.b2, self.t)
        for p, g, m, v, wd in zip(self.vars, grads, self.m, self.v, self.wd):
            if g is None:
                continue
            g = tf.convert_to_tensor(g)
            p.assign(p * (1.0 - self.lr * wd))
            m.assign(self.b1 * m + (1.0 - self.b1) * g)
            v.assign(self.b2 * v + (1.0 - self.b2) * g * g)
            denom = tf.sqrt(v) / tf.sqrt(bc2) + self.eps
            p.assign_sub((self.lr / bc1) * m / denom)


# ----------------------------------------------------------------------------
# Data + training (mirrors lib/data.py:normalize and bin/ft_transformer.py's loop)
# ----------------------------------------------------------------------------
import json, time
from pathlib import Path
from scipy.special import expit


def load_dataset(dir_):
    """Loads N_{train,val,test}.npy / y_*.npy / info.json written in the official repo's format."""
    dir_ = Path(dir_)
    info = json.loads((dir_ / "info.json").read_text())
    N = {p: np.load(dir_ / f"N_{p}.npy") for p in ["train", "val", "test"]}
    y = {p: np.load(dir_ / f"y_{p}.npy") for p in ["train", "val", "test"]}
    assert info["n_cat_features"] == 0, "this port supports numerical features only (enough for Jannis / Higgs)"
    return N, y, info


def quantile_normalize(N, seed, noise=1e-3):
    """Same as lib.data.normalize(..., 'quantile', seed) incl. the noise trick for the fit."""
    from sklearn.preprocessing import QuantileTransformer
    N = {k: v.copy() for k, v in N.items()}
    if any(np.isnan(v).any() for v in N.values()):  # official: impute with the train mean
        mean = np.nanmean(N["train"], axis=0)
        for v in N.values():
            r, c = np.where(np.isnan(v)); v[r, c] = mean[c]
    X_train = N["train"].copy()
    normalizer = QuantileTransformer(
        output_distribution="normal", n_quantiles=max(min(X_train.shape[0] // 30, 1000), 10),
        subsample=int(1e9), random_state=seed,
    )
    stds = np.std(X_train, axis=0, keepdims=True)
    noise_std = noise / np.maximum(stds, noise)
    X_train += noise_std * np.random.default_rng(seed).standard_normal(X_train.shape)
    normalizer.fit(X_train)
    return {k: normalizer.transform(v).astype("float32") for k, v in N.items()}


def _score(task, y, logits):
    if task == "binclass":
        pred = np.round(expit(logits)).astype("int64")
    else:
        pred = logits.argmax(axis=1)
    return float((pred == y).mean())


def train_ft(N, y, info, model_cfg, train_cfg, seed, max_epochs=10**9, verbose=True):
    """Trains one FT-Transformer; returns metrics at the best-validation epoch."""
    assert model_cfg.get("activation", "reglu") == "reglu" and model_cfg.get("initialization", "kaiming") == "kaiming"
    assert not model_cfg.get("kv_compression")
    keras.utils.set_random_seed(seed)
    task = info["task_type"]; multiclass = task == "multiclass"
    d_out = info["n_classes"] if multiclass else 1
    X = quantile_normalize(N, seed)
    Xt = {p: tf.constant(X[p]) for p in X}
    yt_train = tf.constant(y["train"].astype("int64" if multiclass else "float32"))

    model = FTTransformer(
        d_numerical=X["train"].shape[1], d_out=d_out,
        **{k: model_cfg[k] for k in ["n_layers", "d_token", "n_heads", "d_ffn_factor", "attention_dropout",
                                     "ffn_dropout", "residual_dropout", "prenormalization"]},
    )
    model(Xt["train"][:2])
    P = named_parameters(model)
    variables = [v for _, v, _ in P]
    assert len(variables) == len(model.trainable_variables)
    n_params = int(sum(int(np.prod(v.shape)) for v in variables))
    opt = TorchAdamW(variables, [needs_wd(n) for n, _, _ in P], train_cfg["lr"], train_cfg["weight_decay"])
    bs, patience = train_cfg["batch_size"], train_cfg["patience"]
    n_train = X["train"].shape[0]

    def loss_fn(yb, logits):
        if multiclass:
            return tf.reduce_mean(tf.nn.sparse_softmax_cross_entropy_with_logits(yb, logits))
        return tf.reduce_mean(tf.nn.sigmoid_cross_entropy_with_logits(labels=yb, logits=logits))

    @tf.function(input_signature=[tf.TensorSpec([None], tf.int32)])
    def train_step(idx):
        xb, yb = tf.gather(Xt["train"], idx), tf.gather(yt_train, idx)
        with tf.GradientTape() as tape:
            loss = loss_fn(yb, model(xb, training=True))
        opt.apply(tape.gradient(loss, variables))
        return loss

    @tf.function(reduce_retracing=True)
    def infer(x):
        return model(x, training=False)

    def predict(part):
        x, out, eb = Xt[part], [], train_cfg.get("eval_batch_size", 8192)
        for i in range(0, x.shape[0], eb):
            out.append(infer(x[i:i + eb]).numpy())
        return np.concatenate(out)

    def evaluate():
        res = {}
        for part in ["val", "test"]:
            lg = predict(part); res[part] = _score(task, y[part], lg)
            if task == "binclass":
                from sklearn.metrics import roc_auc_score
                res[part + "_auc"] = float(roc_auc_score(y[part], expit(lg)))
        return res

    best, bad, best_epoch, best_w, best_res, t0 = -np.inf, 0, 0, None, None, time.time()
    for epoch in range(1, max_epochs + 1):
        perm = tf.random.shuffle(tf.range(n_train, dtype=tf.int32))
        losses = [train_step(perm[i:i + bs]) for i in range(0, n_train, bs)]
        res = evaluate()
        if res["val"] > best:                       # success (strict improvement), like zero.ProgressTracker
            best, bad, best_epoch, best_res = res["val"], 0, epoch, res
            best_w = [v.numpy() for v in variables]
        else:
            bad += 1
        if verbose:
            print(f"epoch {epoch:3d} | loss {float(tf.reduce_mean(losses)):.4f} | val {res['val']:.4f} | test {res['test']:.4f}"
                  f" | best val {best:.4f} (ep {best_epoch}) | {time.time() - t0:.0f}s", flush=True)
        if bad > patience:                          # patience + 1 consecutive epochs without improvement
            break
    for v, w in zip(variables, best_w):
        v.assign(w)
    final = evaluate()
    return {"seed": seed, "best_epoch": best_epoch, "val": final["val"], "test": final["test"],
            **({"test_auc": final["test_auc"]} if "test_auc" in final else {}),
            "n_parameters": n_params, "seconds": time.time() - t0}
''')
print('wrote ft_tf.py')

wrote ft_tf.py


## 3. Train the TensorFlow port, one run per seed

In [8]:
import tensorflow as tf, keras
assert int(keras.__version__.split(".")[0]) >= 3, "Keras 3 / TF >= 2.16 required"
sys.path.insert(0, str(Path.cwd()))
import ft_tf
gpus = tf.config.list_physical_devices("GPU")
for g in gpus: tf.config.experimental.set_memory_growth(g, True)
print("TF GPUs:", gpus)

def tf_results_file(ds): return OUT / "tf" / f"{ds}.jsonl"
def tf_done(ds):
    f = tf_results_file(ds)
    return {json.loads(l)["seed"]: json.loads(l) for l in f.read_text().splitlines()} if f.exists() else {}

if RUN_TENSORFLOW_PORT:
    for ds in DATASETS:
        model_cfg, train_cfg = dict(CFG[ds]["model"]), dict(CFG[ds]["training"])
        if SMOKE_TEST: train_cfg["batch_size"] = 128
        N, y, info = ft_tf.load_dataset(DATA / ds)
        tf_results_file(ds).parent.mkdir(parents=True, exist_ok=True)
        for seed in SEEDS:
            if seed in tf_done(ds): continue
            r = ft_tf.train_ft(N, y, info, model_cfg, train_cfg, seed, max_epochs=2 if SMOKE_TEST else 10**9, verbose=False)
            with open(tf_results_file(ds), "a") as f: f.write(json.dumps(r) + "\n")
            print(f"[tf port ] {ds:12s} seed {seed:2d} | test acc {r['test']:.4f} | best epoch {r['best_epoch']} | {r['seconds']/60:.1f} min", flush=True)

TF GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


KeyboardInterrupt: 